# Notebook Pengujian Prediction Request Model Serving

- **Nama**: Sonny Ariady
- **Username Dicoding**: sonnyariady
- **Model Endpoint**: TensorFlow Serving (`http://localhost:8501/v1/models/heart-disease-model:predict`) / Local SavedModel Fallback

Notebook ini digunakan untuk menguji dan melakukan prediction request ke model serving yang telah dideploy.

## 1. Import Library

In [1]:
import os
import json
import base64
import requests
import numpy as np
import tensorflow as tf

print(f"TensorFlow Version: {tf.__version__}")


C:\Latihan\AI\Dicoding\MachineLearningPipeline\venv\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


TensorFlow Version: 2.13.1


## 2. Menyiapkan Sample Data Pasien (Clinical Inputs)

In [2]:
# Sampel 1: Pasien Risiko Tinggi Penyakit Jantung
patient_high_risk = {
    'age': [63],
    'sex': [1],
    'cp': [3],
    'trestbps': [145],
    'chol': [233],
    'fbs': [1],
    'restecg': [0],
    'thalach': [150],
    'exang': [0],
    'oldpeak': [2.3],
    'slope': [0],
    'ca': [0],
    'thal': [1]
}

# Sampel 2: Pasien Sehat (Risiko Rendah)
patient_low_risk = {
    'age': [37],
    'sex': [0],
    'cp': [2],
    'trestbps': [130],
    'chol': [250],
    'fbs': [0],
    'restecg': [1],
    'thalach': [187],
    'exang': [0],
    'oldpeak': [3.5],
    'slope': [0],
    'ca': [0],
    'thal': [2]
}

def create_tf_example(data):
    feature = {}
    for key, val in data.items():
        if isinstance(val[0], float):
            feature[key] = tf.train.Feature(float_list=tf.train.FloatList(value=val))
        elif isinstance(val[0], int):
            feature[key] = tf.train.Feature(int64_list=tf.train.Int64List(value=val))
        elif isinstance(val[0], str):
            feature[key] = tf.train.Feature(bytes_list=tf.train.BytesList(value=[val[0].encode('utf-8')]))
    return tf.train.Example(features=tf.train.Features(feature=feature))

example_high = create_tf_example(patient_high_risk)
example_low = create_tf_example(patient_low_risk)

serialized_high = example_high.SerializeToString()
serialized_low = example_low.SerializeToString()


## 3. Uji Prediksi Menggunakan Direct SavedModel (Local Inference)

In [3]:
serving_model_path = os.path.join('serving_model', 'heart-disease-model')

# Temukan folder versi model terbaru
subdirs = [os.path.join(serving_model_path, d) for d in os.listdir(serving_model_path) if d.isdigit()]
latest_model_path = max(subdirs, key=os.path.getmtime) if subdirs else serving_model_path

print(f"Loading SavedModel from: {latest_model_path}")
loaded_model = tf.saved_model.load(latest_model_path)
infer_fn = loaded_model.signatures['serving_default']

# Run inference
res_high = infer_fn(examples=tf.constant([serialized_high]))
res_low = infer_fn(examples=tf.constant([serialized_low]))

print("\n--- HASIL PREDIKSI LOKAL ---")
prob_high = float(list(res_high.values())[0].numpy()[0][0])
prob_low = float(list(res_low.values())[0].numpy()[0][0])

print(f"Pasien 1 (Risiko Tinggi) -> Probabilitas Penyakit Jantung: {prob_high:.4f} ({'POSITIF' if prob_high > 0.5 else 'NEGATIF'})")
print(f"Pasien 2 (Risiko Rendah) -> Probabilitas Penyakit Jantung: {prob_low:.4f} ({'POSITIF' if prob_low > 0.5 else 'NEGATIF'})")


Loading SavedModel from: serving_model\heart-disease-model\1788423466



--- HASIL PREDIKSI LOKAL ---
Pasien 1 (Risiko Tinggi) -> Probabilitas Penyakit Jantung: 0.8781 (POSITIF)
Pasien 2 (Risiko Rendah) -> Probabilitas Penyakit Jantung: 0.8161 (POSITIF)


## 4. Uji Prediksi via TensorFlow Serving REST API

In [4]:
# URL Endpoint TensorFlow Serving
endpoint_url = "http://localhost:8501/v1/models/heart-disease-model:predict"

# Format payload b64 untuk TF Serving REST API
b64_high = base64.b64encode(serialized_high).decode('utf-8')
b64_low = base64.b64encode(serialized_low).decode('utf-8')

payload = {
    "signature_name": "serving_default",
    "instances": [
        {"b64": b64_high},
        {"b64": b64_low}
    ]
}

try:
    response = requests.post(endpoint_url, json=payload, timeout=5)
    if response.status_code == 200:
        predictions = response.json()['predictions']
        print("--- HASIL PREDIKSI TF SERVING REST API ---")
        for i, pred in enumerate(predictions):
            prob = pred[0]
            print(f"Sampel {i+1} Probabilitas: {prob:.4f} -> Result: {'POSITIF' if prob > 0.5 else 'NEGATIF'}")
    else:
        print(f"HTTP Request Response Code: {response.status_code}")
        print("Detail:", response.text)
except Exception as e:
    print("TF Serving container offline / tidak terhubung. Gunakan uji prediksi lokal di sel 3 di atas.")
    print("Error info:", e)


TF Serving container offline / tidak terhubung. Gunakan uji prediksi lokal di sel 3 di atas.
Error info: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
